# EP01｜開發環境與 NumPy 入門

[![在 Colab 開啟](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chainchen1103/llm-from-scratch/blob/main/notebooks/ep01/ep01_numpy_shapes.ipynb)

這份 notebook 對應網站上的 [EP01 圖文教學](https://chainchen1103.github.io/llm-from-scratch/episodes/ep01/)。開發環境的建置請照網站上的步驟做；這裡專心練習 NumPy：

1. 陣列 array 的形狀 shape、維度、資料型別
2. 索引、切片、改變形狀
3. `axis`：沿著哪個方向做
4. **廣播 broadcasting**：本集重點
5. 從零寫出數值穩定的 Softmax，並用測試檢查
6. 弄壞它：幾個「程式照跑、結果卻錯了」的例子

這份 notebook 只用到 NumPy，Colab 已經內建，不用安裝任何東西。在自己的電腦上，用網站步驟 6 建立的 `llm-labs` 專案開啟即可。

**使用方式**：從上到下依序執行（Shift + Enter）。標著「先猜猜看」的地方，先在心裡猜結果再執行。

In [ ]:
# Colab 已經內建 NumPy，不用安裝；在自己的電腦上，請用網站步驟 6 建立的 llm-labs 環境。
# %pip install -q numpy

import sys

import numpy as np

print("Python", sys.version.split()[0], "| NumPy", np.__version__)

這份 notebook 用 NumPy 2.5 測試過。NumPy 2.3 以前的版本，少數輸出的顯示格式會稍微不同（例如弄壞它 3 的 `float32`），數值是一樣的。

## 1. 為什麼要 NumPy？

比較用 Python 的 list 和用 NumPy，算一百萬個數字的平方和。第一次執行常常比較慢（要先做一些準備），所以每種寫法各跑 5 次，取最快的一次來比。

In [ ]:
import time

n = 1_000_000
xs = [i * 0.001 for i in range(n)]
a = np.array(xs)


def best_of_5(f):
    """執行 f 五次，回傳最快的一次花了幾秒"""
    times = []
    for _ in range(5):
        t0 = time.perf_counter()
        f()
        times.append(time.perf_counter() - t0)
    return min(times)


t_list = best_of_5(lambda: sum(x * x for x in xs))
t_numpy = best_of_5(lambda: (a * a).sum())
print(f"list: {t_list*1000:.1f} ms, NumPy: {t_numpy*1000:.1f} ms，NumPy 快了約 {t_list/t_numpy:.0f} 倍")
print("兩種算法結果一樣嗎？", np.isclose(sum(x * x for x in xs), (a * a).sum()))

NumPy 的運算由編譯好的 C 程式一次處理整個陣列，不用 Python 一個一個算。這種「整批處理、不寫迴圈」的寫法叫**向量化 vectorization**。

## 2. 陣列的三個基本屬性

- `shape`：每個維度有多長
- `ndim`：有幾個維度
- `dtype`：裡面的數字是哪一種型別

In [ ]:
a = np.array([1, 2, 3])
print(a, a.dtype, a.shape, a.ndim)

m = np.array([[1.0, 2.0, 3.0],
              [4.0, 5.0, 6.0]])
print(m.dtype, m.shape, m.ndim)

In [ ]:
print(np.zeros((2, 3)))
print(np.ones(3))
print(np.arange(0, 10, 2))      # 從 0 開始、每次加 2、不到 10
print(np.linspace(0, 1, 5))     # 0 到 1 平均切 5 個點

讀形狀的習慣是**從左到右、從外到內**：`(2, 3)` 念成「2 個、每個長 3」。

**練習**：EP00 的殘差流形狀是 `(1, 4, 896)`。建立一個這個形狀、全部是 0 的陣列，印出它的 `ndim` 和總共有幾個數字（`.size`）。

In [ ]:
x = np.zeros((1, 4, 896))
print(x.ndim, x.size)

## 3. 索引與切片

索引 indexing 是取出某個位置的元素，切片 slicing 是取出一段範圍。位置都從 0 開始數，`-1` 是最後一個，範圍「含頭不含尾」。

In [ ]:
x = np.arange(10)
print(x)
print(x[0], x[-1])
print(x[2:5])
print(x[::2])
print(x[::-1])

In [ ]:
m = np.arange(12).reshape(3, 4)
print(m)
print("m[1, 2] =", m[1, 2])
print("m[1]    =", m[1])
print("m[:, 2] =", m[:, 2])     # 所有列、每列的第 2 個
print("m[0:2, 1:3] =")
print(m[0:2, 1:3])

In [ ]:
x = np.arange(10)
print(x[x % 2 == 0])   # 用條件挑出元素
print(m[[0, 2]])       # 挑出第 0 列和第 2 列

## 4. 改變形狀

`reshape` 不改變資料的順序，只改變「怎麼看」它，元素總數必須一樣。

In [ ]:
x = np.arange(6)
print(x.reshape(2, 3))
print(x.reshape(3, -1).shape)   # -1：這個維度你幫我算
m = x.reshape(2, 3)
print(m.T.shape)                # 轉置
print(m.reshape(-1))            # 攤平回一維
v = np.array([1, 2, 3])
print(v[:, None].shape, v[None, :].shape)

**先猜猜看**：`np.arange(6).reshape(4, 2)` 會發生什麼事？

In [ ]:
try:
    np.arange(6).reshape(4, 2)
except ValueError as e:
    print("ValueError:", e)

## 5. `axis`：沿著哪個方向做

**`axis=k` 就是把第 k 個維度「壓掉」。**`keepdims=True` 會把被壓掉的維度留下來，長度變成 1。

In [ ]:
m = np.array([[1, 2, 3],
              [4, 5, 6]])
print(m.sum())
print(m.sum(axis=0), m.sum(axis=0).shape)
print(m.sum(axis=1), m.sum(axis=1).shape)
print(m.mean(axis=1, keepdims=True), m.mean(axis=1, keepdims=True).shape)
print(m.argmax(axis=1))

**練習**：形狀 `(32, 128, 768)` 的陣列（32 句話、每句 128 個 token、每個 token 長 768），沿 `axis=-1` 取平均後是什麼形狀？加上 `keepdims=True` 呢？先猜再執行。

In [ ]:
x = np.zeros((32, 128, 768))
print(x.mean(axis=-1).shape)
print(x.mean(axis=-1, keepdims=True).shape)

## 6. 本集重點：廣播 broadcasting

兩個形狀不同的陣列做運算時，NumPy 會自動把小的那個「延伸」成大的形狀。

**規則**：把兩個形狀靠右對齊，從最右邊的維度開始逐一比較：

1. 兩個長度相等：可以。
2. 其中一個長度是 1：把它延伸成另一個的長度。
3. 其中一個沒有這個維度：當成長度 1，再套用第 2 條。
4. 以上都不符合：報錯。

In [ ]:
m = np.array([[1.0, 2.0, 3.0],
              [4.0, 5.0, 6.0]])            # (2, 3)
print(m + 10)
print(m + np.array([100, 200, 300]))       # (3,) 延伸成 (2, 3)
print(m - m.mean(axis=1, keepdims=True))   # (2, 1) 延伸成 (2, 3)：每列減掉自己的平均

`np.broadcast_shapes` 可以直接問 NumPy：這兩個形狀廣播之後是什麼形狀？

In [ ]:
for a, b in [((2, 3), (3,)), ((2, 3), (2, 1)), ((2, 3), (2,)), ((3,), (3, 1))]:
    try:
        print(a, b, "→", np.broadcast_shapes(a, b))
    except ValueError:
        print(a, b, "→ 錯誤：不能廣播")

**練習**：形狀 `(128, 768)` 和 `(128,)` 不能相加。把 `b` 改成 `(128, 1)`，讓每一列加上自己的那個數。

In [ ]:
x = np.zeros((128, 768))
b = np.arange(128)
try:
    x + b
except ValueError as e:
    print("ValueError:", e)
y = x + b[:, None]          # 或 b.reshape(-1, 1)
print(y.shape, y[5, :3])    # 第 5 列的每個數都是 5

## 7. `*` 和 `@` 不一樣

`*` 是逐元素相乘，`@` 是矩陣乘法 matrix multiplication。矩陣乘法的形狀規則：`(a, b) @ (b, c)` 得到 `(a, c)`。

In [ ]:
A = np.array([[1, 2], [3, 4]])
B = np.array([[10, 20], [30, 40]])
print(A * B)
print(A @ B)
X = np.ones((3, 4))
W = np.ones((4, 5))
print((X @ W).shape)

## 8. 亂數與 seed

固定種子 seed，同樣的程式就會得到同樣的亂數。之後每個實驗都要固定 seed，結果才能重現。

In [ ]:
rng = np.random.default_rng(42)
print(rng.integers(0, 10, size=5))
rng = np.random.default_rng(42)
print(rng.integers(0, 10, size=5))

## 9. 動手做：從零寫出 Softmax

$$p_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

要求：

1. 只用 NumPy，不寫 Python 迴圈。
2. 支援任意形狀，沿著 `axis` 做（預設 `-1`，也就是最後一個維度）。
3. **數值穩定**：輸入 `[1000, 1001, 1002]` 也不能壞掉。提示：每個分數減掉同一個數，結果不變；每一列減掉**自己的**最大值就不會溢位（用 `axis` 和 `keepdims=True`）。

把下面的 `softmax` 寫完，再執行下一格的檢查。

In [ ]:
def softmax(z, axis=-1):
    """沿著 axis 把分數 z 轉成機率。z 可以是任意形狀。"""
    # 你的實作
    raise NotImplementedError("還沒寫好")

In [ ]:
def check_softmax(softmax):
    """和網站上 tests/test_ep01_softmax.py 一樣的 7 個測試。"""
    def sums_to_one():
        z = np.array([[1.0, 2.0, 3.0], [0.0, 0.0, 0.0]])
        assert np.allclose(softmax(z).sum(axis=-1), 1.0)

    def known_values():
        p = softmax(np.array([1.0, 2.0, 3.0]))
        assert np.allclose(p, [0.09003057, 0.24472847, 0.66524096])

    def large_inputs_do_not_overflow():
        p = softmax(np.array([1000.0, 1001.0, 1002.0]))
        assert np.all(np.isfinite(p))
        assert np.allclose(p, softmax(np.array([0.0, 1.0, 2.0])))

    def temperature_matches_ep00_demo():
        z = np.log(np.array([0.62, 0.21, 0.09, 0.04, 0.04]))
        assert np.allclose(softmax(z / 2.0), [0.405, 0.236, 0.154, 0.103, 0.103], atol=1e-3)

    def shape_is_preserved():
        z = np.zeros((4, 7))
        assert softmax(z).shape == (4, 7)

    def each_row_uses_its_own_max():
        z = np.array([[0.0, 1.0, 2.0], [1000.0, 1001.0, 1002.0]])
        p = softmax(z)
        assert np.all(np.isfinite(p))
        assert np.allclose(p[0], p[1])

    def axis_0():
        z = np.array([[1.0, 2.0, 3.0], [1.0, 1.0, 1.0]])
        assert np.allclose(softmax(z, axis=0).sum(axis=0), 1.0)

    passed = 0
    tests = [sums_to_one, known_values, large_inputs_do_not_overflow, temperature_matches_ep00_demo, shape_is_preserved,
             each_row_uses_its_own_max, axis_0]
    for t in tests:
        try:
            with np.errstate(all="ignore"):
                t()
            print("✓", t.__name__)
            passed += 1
        except NotImplementedError:
            print("… 還沒寫好 softmax")
            return
        except Exception as e:
            print("✗", t.__name__, type(e).__name__, e)
    print(f"{passed}/{len(tests)} 通過")


check_softmax(softmax)

## 10. 弄壞它

每一個都先**猜**結果，再執行，最後用自己的話解釋。解釋在網站的「弄壞它」段落。

### 弄壞它 1：廣播悄悄算錯

你可能以為結果是 `[0 0 0]`。

In [ ]:
a = np.array([1, 2, 3])          # (3,)
b = np.array([[1], [2], [3]])    # (3, 1)
print((a - b).shape)
print(a - b)

沒有任何錯誤訊息，但結果是 `(3, 3)`。防法：在關鍵的地方檢查形狀。

In [ ]:
def subtract(a, b):
    assert a.shape == b.shape, f"形狀不同：{a.shape} 和 {b.shape}"
    return a - b

try:
    subtract(a, b)
except AssertionError as e:
    print("AssertionError:", e)

### 弄壞它 2：Softmax 溢位

In [ ]:
def softmax_naive(z):
    e = np.exp(z)
    return e / e.sum()

print(softmax_naive(np.array([1.0, 2.0, 3.0])))
print(softmax_naive(np.array([1000.0, 1001.0, 1002.0])))
print(np.exp(709.0), np.exp(710.0))

### 弄壞它 3：整數溢位與小數誤差

In [ ]:
u = np.array([200], dtype=np.uint8)            # uint8 只能存 0 到 255
print(u + np.uint8(100))
print(0.1 + 0.2 == 0.3)
print(np.isclose(0.1 + 0.2, 0.3))
print(np.float32(16777216) + np.float32(1))

## 11. 參考解答

先自己寫，卡住再看。

In [ ]:
def softmax(z, axis=-1):
    """沿著 axis 把分數 z 轉成機率。z 可以是任意形狀。"""
    z = np.asarray(z, dtype=float)
    shifted = z - z.max(axis=axis, keepdims=True)   # 每一列減掉自己的最大值
    e = np.exp(shifted)
    return e / e.sum(axis=axis, keepdims=True)      # 每一列除以自己的總和


check_softmax(softmax)

### 弄壞它 4：axis 用錯

如果沿 `axis=0` 做 Softmax，而不是 `axis=1`，每一列的機率加起來還是 1 嗎？

In [ ]:
scores = np.array([[1.0, 2.0, 3.0],     # 第 0 個 token 的 3 個候選分數
                   [1.0, 1.0, 1.0]])    # 第 1 個 token 的 3 個候選分數
print(np.round(softmax(scores, axis=0), 3))
print("每一列的加總：", softmax(scores, axis=0).sum(axis=1))
print(np.round(softmax(scores, axis=1), 3))
print("每一列的加總：", softmax(scores, axis=1).sum(axis=1))

沿 `axis=0` 做，比較的是「同一個候選、在兩個 token 之間」，對語言模型沒有意義。我們要的是每個 token 自己的候選之間的機率，所以要沿最後一個維度做（`axis=-1`）。

## 下一集

EP02 會介紹之後一直會用到的數學符號：Softmax 裡的 $e^x$ 和 $\sum$，以及對數 $\log$。